In [2]:
import tmm

from numpy import pi, inf
import numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
#%matplotlib inline
%matplotlib qt

degree = pi/180

In [3]:
# Lorentzian dielectric function for spatial KK stack
def eps(x, a, gam, nb):
    return nb**2 - a * gam / (x + 1j*gam)

In [42]:
def generate_n_and_d(gam, a, nb, plot_flag):

    dx      = gam/100               # Step size in 'continuous' Lorentzian
    xmin    = -gam * 200           # Limits of Lorentzian
    xmax    = - xmin

    nx      = 1 + int(np.floor((xmax - xmin) / dx))
    xx      = np.linspace(xmin, xmax, nx)
    ee      = eps(xx,a,gam,nb)                    # Smooth Lorentzian curve
    nk      = np.sqrt(ee)

    k_max = np.max(np.imag(nk))     # Max k value. used to set max n-step size 
    del_n   = k_max/25                # Max n-step size in discrete Lorentzian approximation
    del_x   = 15*gam                # Max x-step size in discrete Lorentzian approximation

    xq      = [xx[0]]                               
    nq      = [nb]
    count = 0
    for k in range(0,nx):
        if abs((nk[k]) - (nq[count])) > del_n or abs((xx[k]) - (xq[count])) > del_x:
            xq.append(xx[k])
            nq.append(nk[k])
            count = count + 1

    xq = np.append(xq,xmax)
    nq = np.append(nq,nb)
    midpoints = (xq[:-1] + xq[1:]) / 2
    d_list = np.diff(midpoints)
    n_list  = nq[1:-1]

    #midpoints = np.insert(midpoints,0,xq[0]); midpoints = np.append(midpoints,xq[-1])
    #d_list = np.insert(d_list,0,inf); d_list = np.append(d_list,inf)          # Append super- & sub-strate
    #n_list = np.insert(n_list,0,nb); n_list = np.append(n_list,nb)

    
    # plot imaginary and real part of refractive index
    if (plot_flag):
        plt.figure()
        plt.plot(xx/gam, np.real(nk), label='smooth')
        plt.step(xq/gam, np.real(nq), where='mid', label='discrete')
        plt.plot((midpoints[1:] + midpoints[:-1])/2/gam, np.real(n_list), '*', label='inputs')
        plt.xlabel('x/x0')
        plt.ylabel('Refractive Index')
        plt.title(f'Real Part of Refractive Index from A={a}, x0={gam}')
        #plt.xlim([-5, 5])
        plt.legend(loc='upper right')

        plt.figure()
        plt.plot(xx/gam, np.imag(nk), label='smooth')
        plt.step(xq/gam, np.imag(nq), where='mid', label='discrete')
        plt.plot((midpoints[1:] + midpoints[:-1])/2/gam, np.imag(n_list), '*', label='inputs')
        plt.xlabel('x/x0')
        plt.ylabel('Refractive Index')
        plt.title(f'Imaginary Part of Refractive Index from A={a}, x0={gam}')
        plt.legend(loc='upper right')
        #plt.xlim([-5, 5])

        plt.show()
    

    return (n_list.tolist(), d_list.tolist())

In [5]:
def generate_StackofStacks(gam, a, nb, num_stacks, t_prop):
    n_list = []
    d_list = []
    c_list = []
    c_list_KK = []

    n_list_KK, d_list_KK = generate_n_and_d(gam, a, nb, False)
    t_KK = np.sum(d_list_KK)
    t_inc = t_prop*t_KK
    print(f"t_KK is: {t_KK}")
    print(f"t_inc is: {t_inc}")
    t_total = num_stacks*(t_inc + t_KK)
    print(f"t_total is: {t_total}")
    #print(f'length of KK MS is: {t_KK}')

    k_bulk = np.sum(d_list_KK * np.imag(n_list_KK)) / t_KK
    print(f'k_bulk from np.sum is: {k_bulk}')
    losses_total = t_total * k_bulk
    print(f'total losses are: {losses_total}')

    for i in range(len(d_list_KK)):
        c_list_KK.append('c')

    for i in range(num_stacks):
        n_list.extend(n_list_KK)
        d_list.extend(d_list_KK)
        c_list.extend(c_list_KK)
        n_list.append(nb)
        d_list.append(t_inc)
        c_list.append('i')
    
    d_list.append(inf)
    d_list.insert(0, inf)
    n_list.append(nb)
    n_list.insert(0, nb)
    c_list.insert(0, 'i')
    c_list.append('i')


    return (n_list, d_list, c_list, losses_total)



In [6]:
# wavelength dependence calculation of RTA using incoherent TMM function (inc_tmm)

def TRA_func(n_list, d_list, c_list):
    pol = 'p'
    lambda_list = np.linspace(2,5,100)
    angle = 0
    T_list = np.zeros_like(lambda_list)
    R_list = np.zeros_like(lambda_list)
    A_list = np.zeros_like(lambda_list)
    
    for j, lamb in enumerate(lambda_list):
        T_list[j] = tmm.inc_tmm(pol, n_list, d_list, c_list, angle, lamb)['T']
        R_list[j] = tmm.inc_tmm(pol, n_list, d_list, c_list, angle, lamb)['R']
      #  T_list[j] = tmm.coh_tmm(pol, n_list, d_list, angle, lamb)['T']
      #  R_list[j] = tmm.coh_tmm(pol, n_list, d_list, angle, lamb)['R']
        A_list[j] = 1 - T_list[j] - R_list[j]

    return (T_list, R_list, A_list)
    

In [53]:
# wavelength dependence calculation of RTA using incoherent TMM function (inc_tmm)

def TRA_single_func(n_list, d_list, lamb):
    pol = 'p'
    angle = 0
    T = tmm.coh_tmm(pol, n_list, d_list, angle, lamb)['T']
    R = tmm.coh_tmm(pol, n_list, d_list, angle, lamb)['R']
    A = 1 - T - R

    return (T, R, A)

In [97]:
nkdata_sapphire = np.genfromtxt('lam_um_T_K_Al2O3_no_ko_ne_ke.dat')
kdata_sapphire = nkdata_sapphire[50:451, 3]
ndata_sapphire = nkdata_sapphire[50:451, 2]
lamdata_sapphire = nkdata_sapphire[50:451, 0]

plt.figure()
#plt.plot(lamdata_sapphire, ndata_sapphire, label='n')
plt.plot(lamdata_sapphire, kdata_sapphire, label='k')
plt.xlabel('Wavelength ($\mu$m)')
plt.ylabel('Refractive Index')
plt.title('Refractive Index Data of Sapphire')
#plt.legend(loc='upper right')
#k_bulk = np.average(kdata_sapphire)
#print(k_bulk)



Text(0.5, 1.0, 'Refractive Index Data of Sapphire')

In [52]:
n_list_KK, d_list_KK = generate_n_and_d(0.001, 0.01, 1.7, False)
print(np.sum(d_list_KK))

0.38922500000000004


In [98]:

#k_min = 1.09e-9, for A_min = 1.3e-7, gam_min = 0.001, k = 1e-9
#k_max = 3.12e-4, for A_max = 0.2, gam_max = 0.001, k = 5.05e-4
nb = 1.7
gam = 0.001
#A = 1.3e-7
A_arr = np.logspace(-7, 0, num=1000)
k_bulk_arr = np.zeros_like(A_arr)

for i, A in enumerate(A_arr):
    n_list_KK, d_list_KK = generate_n_and_d(gam, A, nb, False)
    k_bulk_arr[i] = np.sum(d_list_KK * np.imag(n_list_KK)) / np.sum(d_list_KK)

# Create an interpolation function
interp_func = interp1d(k_bulk_arr, A_arr, kind='linear')

# Now, you can directly calculate values of A for new values in K_data

A_interp = interp_func(kdata_sapphire)


Text(0.5, 1.0, 'Imaginary Part of Refractive Index vs. Lorentzian Amplitude')

In [99]:

plt.figure()
plt.loglog(A_arr, k_bulk_arr, label='interpolated points')
plt.loglog(A_interp, kdata_sapphire, '-*', label='data')
plt.xlabel('Amplitude A')
plt.ylabel('Refractive Index k')
plt.title('Imaginary Part of Refractive Index vs. Lorentzian Amplitude')
plt.legend(loc='upper left')



In [91]:
L_total = 5000
gam = 0.001
nb = 1.7
num_stacks_arr = np.arange(0, 1000, 1)
trans_arr = np.empty((len(lamdata_sapphire), len(num_stacks_arr)), dtype=float)
emiss_LR_arr_KK_first = np.empty((len(lamdata_sapphire), len(num_stacks_arr)), dtype=float)
emiss_RL_arr_KK_first = np.empty((len(lamdata_sapphire), len(num_stacks_arr)), dtype=float)
emiss_LR_arr_bulk_first = np.empty((len(lamdata_sapphire), len(num_stacks_arr)), dtype=float)
emiss_RL_arr_bulk_first = np.empty((len(lamdata_sapphire), len(num_stacks_arr)), dtype=float)

for i,lam in enumerate(lamdata_sapphire):
    print(f'lam = {lam}')
    A = A_interp[i]
    n_list_KK, d_list_KK = generate_n_and_d(gam, A, nb, False)
    t_KK = np.sum(d_list_KK)
    print(f"t_KK is: {t_KK}")
    k_bulk = np.sum(d_list_KK * np.imag(n_list_KK)) / t_KK
    print(f'k_bulk from np.sum is: {k_bulk}')
    d_list_KK.append(inf)
    d_list_KK.insert(0, inf)
    n_list_KK.append(nb)
    n_list_KK.insert(0, nb)

    n_list_reversed_KK = n_list_KK[::-1]
    d_list_reversed_KK = d_list_KK[::-1]

    trans_LR_KK, ref_LR_KK, emiss_LR_KK = TRA_single_func(n_list_KK, d_list_KK, lam)
    trans_LR_KK, ref_RL_KK, emiss_RL_KK = TRA_single_func(n_list_reversed_KK, d_list_reversed_KK, lam)

    for j,N in enumerate(num_stacks_arr):
        
        trans_total_KK = trans_LR_KK**N
        geom_series = 0
        for l in range(N):
            geom_series = geom_series + trans_LR_KK**l
            
        emiss_LR_total_KK = emiss_LR_KK*geom_series
        emiss_RL_total_KK = emiss_RL_KK*geom_series

        #######################################################################

        L_bulk = L_total - N*t_KK
        #print(f"t_total is: {t_total}")
        losses_bulk = L_bulk * k_bulk
        #print(f'total losses are: {losses_total}')

        trans_bulk = np.exp(-4*np.pi*losses_bulk/lam)
        emiss_bulk = 1 - trans_bulk

        trans_arr[i][j] = trans_bulk*trans_total_KK
        emiss_LR_arr_KK_first[i][j] = emiss_LR_total_KK*trans_bulk + emiss_bulk
        emiss_RL_arr_KK_first[i][j] = emiss_RL_total_KK*trans_bulk + emiss_bulk
        emiss_LR_arr_bulk_first[i][j] = emiss_bulk*trans_total_KK + emiss_LR_total_KK
        emiss_RL_arr_bulk_first[i][j] = emiss_bulk*trans_total_KK + emiss_RL_total_KK
        


lam = 2.0
t_KK is: 0.389235
k_bulk from np.sum is: 1.0899999999999998e-09
lam = 2.01
t_KK is: 0.389235
k_bulk from np.sum is: 1.13e-09
lam = 2.02
t_KK is: 0.389235
k_bulk from np.sum is: 1.1699999999999995e-09
lam = 2.03
t_KK is: 0.389235
k_bulk from np.sum is: 1.2099999999999995e-09
lam = 2.04
t_KK is: 0.389235
k_bulk from np.sum is: 1.249999999999999e-09
lam = 2.05
t_KK is: 0.389235
k_bulk from np.sum is: 1.3000000000000003e-09
lam = 2.06
t_KK is: 0.389235
k_bulk from np.sum is: 1.3399999999999989e-09
lam = 2.07
t_KK is: 0.389235
k_bulk from np.sum is: 1.39e-09
lam = 2.08
t_KK is: 0.389235
k_bulk from np.sum is: 1.4399999999999994e-09
lam = 2.09
t_KK is: 0.389235
k_bulk from np.sum is: 1.4999999999999994e-09
lam = 2.1
t_KK is: 0.389235
k_bulk from np.sum is: 1.56e-09
lam = 2.11
t_KK is: 0.389235
k_bulk from np.sum is: 1.6199999999999996e-09
lam = 2.12
t_KK is: 0.389235
k_bulk from np.sum is: 1.6799999999999987e-09
lam = 2.13
t_KK is: 0.389235
k_bulk from np.sum is: 1.7499999999999987

In [92]:
np.save("C:\\Users\\kl89\\MS Window Project\\trans_arr_StackofStacks_num1~10000_lam2~6_gam0-001.npy", trans_arr)
np.save("C:\\Users\\kl89\\MS Window Project\\emiss_LR_arr_KK_first_StackofStacks_num1~10000_lam2~6_gam0-001.npy", emiss_LR_arr_KK_first)
np.save("C:\\Users\\kl89\\MS Window Project\\emiss_RL_arr_KK_first_StackofStacks_num1~10000_lam2~6_gam0-001.npy", emiss_RL_arr_KK_first)
np.save("C:\\Users\\kl89\\MS Window Project\\emiss_LR_arr_bulk_first_StackofStacks_num1~10000_lam2~6_gam0-001.npy", emiss_LR_arr_bulk_first)
np.save("C:\\Users\\kl89\\MS Window Project\\emiss_LR_arr_bulk_first_StackofStacks_num1~10000_lam2~6_gam0-001.npy", emiss_LR_arr_bulk_first)

In [93]:
# Plot enhancement as a colorplot

plt.figure()
plt.imshow((trans_arr/emiss_RL_arr_KK_first).T, interpolation='none', aspect='auto', origin='lower', extent=[lamdata_sapphire[0], lamdata_sapphire[-1],
    num_stacks_arr[0], num_stacks_arr[-1]])
plt.colorbar(label="T/E")
plt.ylabel('Number of Stacks')
plt.xlabel('wavelength (um)')
plt.title('FOM')

Text(0.5, 1.0, 'FOM')

In [102]:
# Plot enhancement against num stacks for set A

N_set = 5000
plt.figure(1)
plt.plot(lamdata_sapphire, trans_arr[:,N_set]/emiss_RL_arr_KK_first[:,N_set], label=f'N={N_set}')
plt.ylabel('FOM')
plt.xlabel('wavelength (um)')
plt.title(f'FOM')
plt.legend(loc='upper right')

# Plot enhancement against num stacks for set A

lam_set = 200
plt.figure(2)
plt.plot(num_stacks_arr, trans_arr[lam_set,:]/emiss_LR_arr_bulk_first[lam_set,:], label=f'lam={lamdata_sapphire[lam_set]}um')
plt.ylabel('FOM')
plt.xlabel('Number of Stacks N')
plt.title(f'FOM')
plt.legend(loc='upper left')

In [ ]:
# sweep over number of layers N and calculate FOM for each stack

#A = 0.1
A_arr = np.arange(0.1, 10, 0.1)
gam = 0.001
#gam_arr = np.arange(0.0001, 0.0011, 0.0001)
#print(gam_arr)
nb = 1.7
t_prop = 0
num_stacks_arr = np.arange(1, 1000, 1)
length = len(num_stacks_arr)
FOM_LR_arr_analytic = np.empty((len(A_arr), length), dtype=float)
FOM_RL_arr_analytic = np.empty((len(A_arr), length), dtype=float)
FOM_bulk_arr_analytic = np.empty((len(A_arr), length), dtype=float)

for i, A in enumerate(A_arr):
    print(f'A is: {A}')

    n_list_KK, d_list_KK = generate_n_and_d(gam, A, nb, False)
    c_list_KK = []
    for m in range(len(d_list_KK)):
        c_list_KK.append('c')

    t_KK = np.sum(d_list_KK)
    t_inc = t_prop*t_KK
    print(f"t_KK is: {t_KK}")
    print(f"t_inc is: {t_inc}")
    k_bulk = np.sum(d_list_KK * np.imag(n_list_KK)) / t_KK
    print(f'k_bulk from np.sum is: {k_bulk}')

    d_list_KK.append(inf)
    d_list_KK.insert(0, inf)
    n_list_KK.append(nb)
    n_list_KK.insert(0, nb)
    c_list_KK.insert(0, 'i')
    c_list_KK.append('i')
    
    n_list_reversed_KK = n_list_KK[::-1]
    d_list_reversed_KK = d_list_KK[::-1]
    c_list_reversed_KK = c_list_KK[::-1]


    T_list_LR_KK, R_list_LR_KK, A_list_LR_KK = TRA_func(n_list_KK, d_list_KK, c_list_KK)
    T_list_RL_KK, R_list_RL_KK, A_list_RL_KK = TRA_func(n_list_reversed_KK, d_list_reversed_KK, c_list_reversed_KK)

    for j, n in enumerate(num_stacks_arr):
        #print(f'num of stacks is: {n}')

        t_total = n*(t_inc + t_KK)
        #print(f"t_total is: {t_total}")
        losses_total = t_total * k_bulk
        #print(f'total losses are: {losses_total}')
        
        T_list_analytic = T_list_LR_KK**n
        geom_series = np.zeros_like(A_list_LR_KK)
        for l in range(n):
            geom_series = geom_series + T_list_LR_KK**l
            
        A_list_LR_analytic = A_list_LR_KK*geom_series
        A_list_RL_analytic = A_list_RL_KK*geom_series

        #######################################################################

        lambda_list = np.linspace(2,5,100)
        delta_lamb = lambda_list[-1] - lambda_list[0]
        trans_bulk = np.exp(-4*np.pi*losses_total/lambda_list)
        emiss_bulk = 1 - trans_bulk

        FOM_LR_arr_analytic[i][j] = (np.trapz(T_list_analytic, x=lambda_list))**2 / np.trapz(A_list_LR_analytic, x=lambda_list) / delta_lamb
        FOM_RL_arr_analytic[i][j] = (np.trapz(T_list_analytic, x=lambda_list))**2 / np.trapz(A_list_RL_analytic, x=lambda_list) / delta_lamb
        FOM_bulk_arr_analytic[i][j] = (np.trapz(trans_bulk, x=lambda_list))**2 / np.trapz(emiss_bulk, x=lambda_list) / delta_lamb

    

In [137]:
np.save("C:\\Users\\kl89\\MS Window Project\\FOM_bulk_analytic_fixed_StackofStacks_num1~10000_A1~100_gam0-001.npy", FOM_bulk_arr_analytic)
np.save("C:\\Users\\kl89\\MS Window Project\\FOM_LR_analytic_fixed_StackofStacks_num1~10000_A1~100_gam0-001.npy", FOM_LR_arr_analytic)
np.save("C:\\Users\\kl89\\MS Window Project\\FOM_RL_analytic_fixed_StackofStacks_num1~10000_A1~100_gam0-001.npy", FOM_RL_arr_analytic)

In [152]:
# Plot enhancement as a colorplot

plt.figure()
plt.imshow(np.log(FOM_RL_arr_analytic[:10,:]/FOM_bulk_arr_analytic[:10,:]), interpolation='none', aspect='auto', origin='lower')
plt.colorbar(label="log(FoM)")
plt.ylabel('Amplitude A')
plt.xlabel('Number of Stacks')
plt.title('Log of FoM Enhancement (gamma=0.001)')

Text(0.5, 1.0, 'Log of FoM Enhancement (gamma=0.001)')

In [100]:
# Plot enhancement against num stacks for set A

A_set = 100
print(FOM_RL_arr_analytic[A_set, :])
print(FOM_bulk_arr_analytic[A_set, :])
plt.figure()
plt.plot(num_stacks_arr, np.log(FOM_RL_arr_analytic[A_set, :] / FOM_bulk_arr_analytic[A_set, :]), '*-')
plt.ylabel('Enhancement')
plt.xlabel('Number of Stacks N')
plt.title(f'FoM Enhancement (gamma=0.001), A={A_set}')

IndexError: index 100 is out of bounds for axis 0 with size 99

In [ ]:
# Plot enhancement against A for set num stacks

num_stacks_set = 1000

plt.figure()
plt.plot(A_arr, np.log(FOM_RL_arr_analytic[:, num_stacks_set] / FOM_bulk_arr_analytic[:, num_stacks_set]), '*-')
plt.ylabel('Enhancement')
plt.xlabel('Amplitude A')
plt.title(f'FoM Enhancement (gamma=0.001), num_stacks={num_stacks_set}')

In [129]:
# Color Plot of Number of Stacks vs. FOM and A

plt.figure()
#plt.imshow(np.log(FOM_RL_arr_analytic/FOM_bulk_arr_analytic), interpolation='none', aspect='auto', origin='lower')
ratio = np.log(FOM_RL_arr_analytic / FOM_bulk_arr_analytic)

# Create arrays for x (i values), y (FOM[i][j] values), and color (j values)
x = np.repeat(np.arange(ratio.shape[0]), ratio.shape[1])  # Repeat i for each j
y = ratio.flatten()  # Flatten the array to get the FOM[i][j] values
colors = np.tile(np.arange(ratio.shape[1]), ratio.shape[0])  # Repeat j for each i

# Plot using scatter with colors representing j
plt.scatter(x, y, c=colors, cmap='viridis', marker='.')
plt.colorbar(label="Number of Stacks")  # Colorbar shows the index j

# Set labels
plt.xlabel("Amplitude A")
plt.ylabel("Log of FoM Enhancement")
plt.title("Number of Stacks Needed Given an FoM and Amplitude")

Text(0.5, 1.0, 'Number of Stacks Needed Given an FoM and Amplitude')

In [128]:
############################ SINGLE CONTOUR PLOT CODE ###########################

plt.figure()
# Assuming FOM_RL_arr_analytic and FOM_bulk_arr_analytic are your 2D arrays
FOM = np.log(FOM_RL_arr_analytic / FOM_bulk_arr_analytic)

# Interpolate FOM[i][j] for a specific FOM value (e.g., FOM = 2)
desired_FOM = 2
# Initialize a list to store the N values for the desired FOM
N_values_for_FOM = []

# Iterate through each value of A and find the closest N that gives the desired FOM
for a in A_arr:
    # Find the index of the closest FOM value to the desired FOM
    idx = np.argmin(np.abs(FOM[int((a - A_arr[0]) / (A_arr[1] - A_arr[0]))] - desired_FOM))
    N_values_for_FOM.append(num_stacks_arr[idx])

plt.plot(A_arr, N_values_for_FOM)
plt.xlabel("Amplitude A")
plt.ylabel("Number of stacks N")
plt.title(f"log(FoM) = {desired_FOM}")
plt.grid(True)
plt.show()


In [126]:
#################### CONTOUR PLOT CODE ##############################

# Assuming FOM_RL_arr_analytic and FOM_bulk_arr_analytic are your 2D arrays
FOM = np.log(FOM_RL_arr_analytic / FOM_bulk_arr_analytic)

# Create a meshgrid for the contour plot
A_grid, N_grid = np.meshgrid(A_arr, num_stacks_arr)

FOM_transposed = FOM.T

# Create an inverted contour plot: contour levels will correspond to N values
plt.figure(figsize=(10, 8))

# Use 'contour' to plot the contours, and we set levels to correspond to specific N values
# We will create the contours for FOM values, but the color scale will correspond to N values.
# We assume FOM[i, j] corresponds to FOM at (A[i], N[j])

# Plot contours for specific FOM levels (can be adjusted as needed)
print(np.min(FOM_transposed))
contour = plt.contour(A_grid, N_grid, FOM_transposed, levels=np.linspace(2, np.max(FOM_transposed), 20), cmap='viridis')

# Add labels and title
plt.xlabel('Amplitude (A)')
plt.ylabel('Number of Stacks (N)')
plt.title('Contour Plot of Constant FoM Enhancement')

# Optional: Add a color bar to show the mapping between contour levels and N values
plt.colorbar(contour, label="Log(FoM)")

plt.grid(True)
plt.show()

0.003484197102164073


In [9]:
for i, A in enumerate(A_arr):
    print(FOM_RL_arr[i][0]/FOM_bulk_arr[i][0])
    print(FOM_RL_arr[i][1]/FOM_bulk_arr[i][1])
    plt.plot(num_stacks_arr, FOM_RL_arr[i]/FOM_bulk_arr[i], '-*', label=f'A={A}')
plt.title('FoM Enhancement (x0 = 0.0005)')
plt.xlabel('Number of Stacks')
plt.ylabel('Enhancement')
plt.legend(loc='upper right')

1.0007050266240705
1.0007043086154523
1.001133790222959
1.0011323663431653
1.0015668604261156
1.001564946591341
1.0020042248924506
1.0020023766379318
1.0024458687768019
1.0024449761121026


In [ ]:
plt.plot(FOM_RL_arr/FOM_bulk_arr, label='RL to bulk')
#plt.plot(FOM_RL_arr/FOM_bulk_arr, label='RL to bulk')
#plt.plot(FOM_RL_arr/FOM_bulk_arr, label='RL to bulk')
#plt.plot(FOM_LR_arr/FOM_bulk_arr, label='LR to bulk')
#plt.plot(FOM_RL_arr/FOM_LR_arr, label='RL to LR')
plt.title('Ratio of FOMs')
plt.xlabel('Number of Stacks')
plt.ylabel('Ratio')
plt.legend()
print(FOM_RL_arr)

In [45]:
np.save("C:\\Users\\kl89\\MS Window Project\\FOM_bulk_StackofStacks_num1~50_A1~5_gam0-0001~0-001.npy", FOM_bulk_arr)
np.save("C:\\Users\\kl89\\MS Window Project\\FOM_LR_StackofStacks_num1~50_A1~5_gam0-0001~0-001.npy", FOM_LR_arr)
np.save("C:\\Users\\kl89\\MS Window Project\\FOM_RL_StackofStacks_num1~50_A1~5_gam0-0001~0-001.npy", FOM_RL_arr)

In [ ]:
# only for when sweeping A, gam, and num_stacks

for i, gam in enumerate(gam_arr): 
    plt.figure(i)
    for j, A in enumerate(A_arr):
        plt.plot(num_stacks_arr, FOM_RL_arr[j, i, :], '*-', label=f'KK, A={A}')
        plt.plot(num_stacks_arr, FOM_bulk_arr[j, i, :], '*-', label=f'bulk, A={A}')
    plt.title(f"FOM Enhancement (x0 = {round(gam, 5)})")
    plt.legend(loc='upper right')
    plt.xlabel("number of stacks")
    plt.ylabel('enhancement')

In [160]:
arr = np.array([1, 3, 5, 7, 9])

midpoints = (arr[:-1] + arr[1:]) / 2

print(arr[0:-1]) 

[1 3 5 7]
